# Solutions · Chapter 02-08 · A full exploratory analysis, and the document it produces

Attempt each exercise before reading. Several of these produce findings the chapter did not - E7 in
particular turns up the strongest aggregation effect in the whole module, on real data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True).frame
households = (housing["Population"] / housing["AveOccup"]).round().astype(int)
suspect = (housing["AveRooms"] > 20) | (housing["AveOccup"] > 20)
capped = housing["MedHouseVal"] == housing["MedHouseVal"].max()

print("loaded:", housing.shape, "| suspect rows:", int(suspect.sum()), "| capped rows:", int(capped.sum()))

## E1 · Three defects in a dataset with nothing missing

1. **A censored target** (02-03, 02-05). 965 rows sit exactly at $500,001; values above were clipped
   to the cap rather than recorded or dropped.
2. **A unit-of-observation mismatch** (02-01). A row is a block group, and its columns are averages
   over three different denominators - block group, household, house.
3. **Ratio artefacts from tiny denominators** (02-05). `AveOccup` reaching 1243 and `AveRooms`
   reaching 142 are arithmetically correct averages over a handful of households.

Others that count: censored `HouseAge` and `MedInc` (02-05); no city column, so every correlation is
pooled (02-06); and nothing about renters, vacancies or the people themselves (02-03).

The pattern is that missing-value and duplicate checks look for **storage** faults, and every defect
above is a **meaning** fault. No automated check finds these, because finding them requires knowing
what the numbers were supposed to be.

## E2 · Why `AveOccup = 1243` is not a data-entry error

Because it is exactly consistent with the other columns in its own row: population 7,460 over 6
households is 1243.33, and the household count itself was recovered from the data to eight decimal
places. A typo would break that consistency; this does not.

What is wrong is the **definition**, not the value. `Population` counts every resident, including
people the census places in group quarters - dormitories, barracks, prisons, care homes - who by
definition have no household. `AveOccup` divides the first by the second, so wherever the two
populations diverge the ratio stops describing anything. The number is a correct answer to a
question nobody wanted to ask.

## E3 · What `HouseAge = 52` means

It means **"52 years old or older"** - the value was clipped, so 52 stands for the entire open-ended
range above it. Since the data is from 1990, that is "built in 1938 or earlier", and 1,273 block
groups - 6.17% - carry it.

Storing it as the number 52 causes three problems:

- **Every model treats it as a measurement.** A linear model reads the step from 51 to 52 as one
  year, when it is really the step from "51 years old" to "somewhere between 52 and 200 years old".
- **Any average is wrong**, and wrong in a known direction: the mean house age is an underestimate.
- **It is silent.** Nothing raises a warning, no summary looks unusual, and the column keeps working.

The honest treatment is to carry an extra boolean column, `house_age_censored`, so the model can
learn that those rows are different - the same indicator trick that beat plain imputation in 02-04.

## E4 · Households, rooms, and what the average is really measuring

In [ ]:
population, ave_occup, ave_rooms = 1200, 2.5, 5.2
n_households = population / ave_occup
print("households  : %.0f" % n_households)
print("total rooms : %.0f" % (ave_rooms * n_households))

print()
print("Now with 900 of the 1,200 residents in a dormitory:")
private_residents = population - 900
print("  people in private households : %d" % private_residents)
print("  true people per household    : %.2f" % (private_residents / n_households))
print("  reported AveOccup            : %.2f" % ave_occup)

480 households and 2,496 rooms.

With 900 people in a dormitory, the 480 households actually contain 300 people - **0.62 per
household**, not 2.5. The reported `AveOccup` is measuring *"everybody in the area divided by the
number of private homes"*, which is a quantity with no meaning: its numerator and denominator count
different populations.

That is the entire mechanism behind the 1243 row, at a scale where it is obvious. The dataset's worst
rows are just this arithmetic with the dormitory much larger relative to the neighbourhood.

## E5 · How much the cap costs the mean

In [ ]:
reported_mean = housing["MedHouseVal"].mean()
assumed_true = 7.2                       # $720,000 for the censored block groups
adjusted = (housing.loc[~capped, "MedHouseVal"].sum() + capped.sum() * assumed_true) / len(housing)

print("reported mean target      : %.4f  ($%s)" % (reported_mean, format(round(reported_mean * 100000), ",")))
print("mean if capped truly 7.2  : %.4f  ($%s)" % (adjusted, format(round(adjusted * 100000), ",")))
print("understatement            : %.4f  ($%s)" % (adjusted - reported_mean,
                                                   format(round((adjusted - reported_mean) * 100000), ",")))

About **0.103 in target units, or roughly $10,300** on a reported mean of $206,900 - a 5%
understatement.

**Why it is a lower bound.** Three reasons, all pushing the same way:

1. $720,000 is an assumption, and the true mean of the censored group is unknown. It could be far
   higher - the top of the California market in 1990 did not stop at $720,000, and the censored group
   has no upper limit at all.
2. The same censoring applies to `HouseAge` and `MedInc`, so other summaries are shifted too.
3. Nothing here accounts for block groups that would have been included had the survey reached them.

The honest sentence is *"the reported mean understates by at least 5%, and by an unknown amount that
depends entirely on how expensive the censored areas really were."*

## E6 · `ceiling_check`

In [ ]:
def ceiling_check(df, column, threshold=0.01):
    values = df[column]
    top = values.max()
    at_top = int((values == top).sum())
    share = at_top / len(values)
    second = np.sort(values.unique())[-2]
    at_second = int((values == second).sum())
    flag = "  <-- LIKELY CEILING" if share > threshold else ""
    print("%-12s max %12.5f  at max %5d (%5.2f%%)   2nd %12.5f  at 2nd %4d%s"
          % (column, top, at_top, 100 * share, second, at_second, flag))
    return share > threshold


flagged = [c for c in housing.columns if ceiling_check(housing, c)]
print()
print("flagged:", flagged)

**`HouseAge` (6.17%) and `MedHouseVal` (4.68%) flag. `MedInc` does not** - 49 rows is 0.24%, below
the 1% threshold, even though it is unmistakably a ceiling: 49 rows at exactly 15.0001 with the next
value at 15.00000 held by 2 rows.

That is the exercise's real lesson. **The rule catches the two large ceilings and misses a real one**,
because a threshold on a share cannot distinguish "few rows at the cap" from "no cap". The second
column in the report is what gives it away: a genuine maximum has one row at the top and one at the
next value down (see `AveRooms`, `AveOccup`, `Population`), whereas a ceiling has a crowd at the top
and a cliff below it.

Which is the general shape of automated data checks - they are prompts to look, and the moment you
treat one as a verdict it starts hiding things. Reading the printed table beats reading `flagged`.

## E7 · Building a region column, and what it does to `HouseAge`

The dataset has no city column, so cluster the coordinates. Eight clusters on raw latitude and
longitude is crude - a degree of latitude and a degree of longitude are not the same distance, and
nothing here is scaled - but it is enough to separate the Bay Area from Los Angeles from the Central
Valley, which is all the exercise needs. Module 08 does clustering properly.

In [ ]:
from sklearn.cluster import KMeans

coords = housing[["Latitude", "Longitude"]].to_numpy()
regions = KMeans(n_clusters=8, n_init=10, random_state=0).fit_predict(coords)

with_region = housing.copy()
with_region["region"] = regions

overall = housing["HouseAge"].corr(housing["MedHouseVal"])
print("r(HouseAge, MedHouseVal) across all of California : %+.3f" % overall)
print()

rows = []
for region_id, block in with_region.groupby("region"):
    rows.append({
        "region": region_id,
        "n": len(block),
        "centre lat": round(float(block["Latitude"].mean()), 2),
        "centre lon": round(float(block["Longitude"].mean()), 2),
        "r(HouseAge, value)": round(float(block["HouseAge"].corr(block["MedHouseVal"])), 3),
        "r(MedInc, value)": round(float(block["MedInc"].corr(block["MedHouseVal"])), 3),
        "mean value": round(float(block["MedHouseVal"].mean()), 2),
    })
by_region = pd.DataFrame(rows)
print(by_region.to_string(index=False))

signs = by_region["r(HouseAge, value)"]
print()
print("regions with a positive HouseAge relationship: %d" % (signs > 0).sum())
print("regions with a negative HouseAge relationship: %d" % (signs < 0).sum())

### This is the strongest aggregation effect in the module, and it is real

Statewide, `r(HouseAge, MedHouseVal)` is **+0.106**: older housing goes with higher value.

**Within regions, six of the eight are negative**, three of them strongly - **-0.335, -0.324 and
-0.278**. Only two are positive, and both are near zero (+0.065 and +0.009). The pooled number does
not merely weaken inside regions; it points the other way from most of them.

The mechanism is 02-06's exactly. Old housing stock is concentrated in the coastal metropolitan
areas, which are the expensive places for reasons that have nothing to do with house age. Pooling
compares a 1930s house in San Francisco with a 1980s house in the Central Valley and reads the
difference between the cities as an effect of age. Inside any one region, where location is roughly
held constant, older houses are worth *less* - which is what anyone who has bought a house would
expect.

**Contrast `r(MedInc, value)`**, which runs from 0.511 to 0.754 across the eight regions and is
0.689 pooled. That relationship survives disaggregation. It is the difference between a finding and
an artefact of mixing populations, and you cannot tell which you have without doing this check.

**What it means for the pricing model:** any model without a location variable will learn "older is
worth more" from the pooled data and apply it inside cities, where it is wrong. This is the single
most important thing found anywhere in the chapter, and it took a column that does not exist in the
file.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))
left.scatter(housing["Longitude"], housing["Latitude"], c=regions, s=3, alpha=0.5, cmap="tab10")
left.set_title("eight regions from coordinates")
left.set_xlabel("longitude")
left.set_ylabel("latitude")

order = by_region.sort_values("r(HouseAge, value)")
colours = ["#D55E00" if v < 0 else "#0072B2" for v in order["r(HouseAge, value)"]]
right.barh([str(r) for r in order["region"]], order["r(HouseAge, value)"], color=colours)
right.axvline(overall, color="black", linestyle="--", linewidth=1.2)
right.text(overall + 0.01, 0.2, f"pooled {overall:+.3f}", fontsize=9)
right.axvline(0, color="grey", linewidth=0.8)
right.set_xlabel("r(HouseAge, MedHouseVal) within region")
right.set_ylabel("region")
right.set_title("pooled says +, most regions say -")
plt.tight_layout()
plt.show()

## E8 · Which correlation moves most, and which changes rank most

In [ ]:
all_rows = housing.corr(numeric_only=True)["MedHouseVal"].drop("MedHouseVal")
sane_rows = housing.loc[~suspect].corr(numeric_only=True)["MedHouseVal"].drop("MedHouseVal")

moves = pd.DataFrame({"all rows": all_rows.round(3), "sane rows": sane_rows.round(3)})
moves["absolute move"] = (moves["sane rows"] - moves["all rows"]).abs().round(3)
moves["rank before"] = moves["all rows"].abs().rank(ascending=False).astype(int)
moves["rank after"] = moves["sane rows"].abs().rank(ascending=False).astype(int)
moves["places gained"] = moves["rank before"] - moves["rank after"]

print(moves.sort_values("absolute move", ascending=False).to_string())

**`AveOccup` wins both.** Its correlation moves **0.218** (from -0.024 to -0.242), and it climbs
**five places, from eighth of eight to third**, on the strength of setting aside 0.38% of rows.

`AveRooms` moves second-most in absolute terms (0.122) but does not move at all in rank - it was
already second and stays second. The two questions genuinely have different answers, which is why the
exercise asks both: absolute movement measures how wrong the number was, rank movement measures
whether the wrongness would have changed a decision.

For anyone shortlisting columns by correlation, only the rank column matters, and by that measure the
79 rows cost exactly one column - the one that would have been discarded.

## E9 · Three reasons to refuse the house-pricing proposal

Ordered by how badly each breaks the proposal:

1. **The target is not a house price.** It is the median value of all houses in a block group of
   600-3,000 people. A model fitted here predicts an area's median, and no amount of accuracy on that
   task makes it a house valuer. This is fatal and no amount of extra data fixes it - it is the wrong
   target. (02-01)
2. **The data is from 1990.** Any price relationship it encodes belongs to a market that has since
   changed beyond recognition. Even as an area-median model it is a historical artefact. Also fatal,
   and fixable only by getting current data. (02-02)
3. **The target is censored at $500,001, and the error metric hides it.** The model has never seen a
   value above the cap and cannot produce one, so it fails exactly where an expensive house most
   needs pricing. A mean absolute error of $52,000 is an average over 20,640 areas, most of them
   cheap; it says nothing about performance at the top. (02-03, 02-07)

A fourth, from E7: without a location variable the model learns "older houses are worth more", which
reverses inside every major city.

The order matters in the room. Reason 1 ends the discussion; reasons 2 and 3 are why it cannot be
rescued by tuning.

## E10 · Why non-random censoring is worse

If the capped 4.68% were a random sample, censoring would remove information evenly and cost
precision - the fitted relationship would stay roughly right, just noisier, and the bias would be a
small predictable shrink.

Because the capped rows are the richest block groups (mean income 7.83 against 3.68), the censoring
instead removes information **specifically from the top of the range** - so the model is least
informed exactly where it is asked to extrapolate, the relationship is measured over a truncated
span, and every summary of the target is biased downward by an amount that depends on the unknown
values. It is 02-04's not-at-random missingness with the values clipped rather than deleted.

## E11 · What is wrong with the three-sigma sweep

**First: the three-sigma rule is computed from a standard deviation that the extreme values
themselves inflated.** `AveOccup` has a mean of 3.07 and a standard deviation of 10.39 - nearly all
of that spread comes from a handful of rows. Three sigma is therefore about 34, and the row at
1243 drags the threshold up so far that genuinely odd rows at 25 or 30 pass it unflagged. This is the
masking effect from 02-05, and here it is severe.

**Second: applied across the whole frame it removes rows for the wrong reasons.** A block group can
be flagged for having a large `Population`, which is not a defect at all, while the actual defect -
the mismatch between residents and households - is a *relationship between two columns* that no
per-column filter can see. The 79 rows worth setting aside were identified by understanding the
ratio, not by measuring distance from a mean.

**And the report is not checkable.** "It removed the bad ones" cannot be verified, because nobody
wrote down what makes a row bad. Reproducing the procedure below removes 846 rows rather than the
1,142 reported, so the analyst's exact variant is unknown - which is itself the problem.

What the reproduction does show is worth more than the argument:

In [ ]:
occup = housing["AveOccup"]
print("AveOccup: mean %.3f  sd %.3f  -> three-sigma cut-off at %.1f"
      % (occup.mean(), occup.std(), occup.mean() + 3 * occup.std()))
print("rows above that cut-off: %d" % (occup > occup.mean() + 3 * occup.std()).sum())
print("rows we actually wanted to set aside on this column: %d" % (occup > 20).sum())
print()
without_worst = occup[occup < 100]
print("recomputed without the rows above 100: sd %.3f -> cut-off %.1f"
      % (without_worst.std(), without_worst.mean() + 3 * without_worst.std()))

In [ ]:
z_scores = (housing - housing.mean()) / housing.std()
swept = (z_scores.abs() > 3).any(axis=1)

print("rows removed by a whole-frame three-sigma sweep: %d (%.2f%%)" % (swept.sum(), 100 * swept.mean()))
print()
print("flagged per column:")
print((z_scores.abs() > 3).sum().to_string())
print()
print("three-sigma cut-off on the TARGET : %.3f   (its maximum is %.5f)"
      % (housing["MedHouseVal"].mean() + 3 * housing["MedHouseVal"].std(),
         housing["MedHouseVal"].max()))
print()
print("of the 79 artefact rows, removed  : %d" % swept[suspect].sum())
print("of the 965 censored rows, removed : %d" % swept[capped].sum())
print("mean target of removed rows %.2f  vs kept %.2f"
      % (housing.loc[swept, "MedHouseVal"].mean(), housing.loc[~swept, "MedHouseVal"].mean()))

Three things in that output, none of them good:

- **The target flags nothing.** Its three-sigma cut-off is 5.530 and its maximum is 5.00001, so the
  censoring has pulled the extreme values *below* the threshold designed to catch them. A filter
  cannot flag values that were already clipped. The most serious defect in the dataset is invisible
  to the procedure meant to find defects.
- **It removes 846 rows to catch 77 of the 79 artefacts.** The other 769 are ordinary block groups
  flagged for having a high income or a large population - neither of which is a defect.
- **The removed rows are the expensive ones**: mean target 3.07 against 2.03 for those kept, and 295
  of the 965 censored rows go with them. The sweep quietly deletes the top of the market, which is
  the part the pricing model most needed.

So the procedure misses the worst problem, solves a real one inefficiently, and introduces a new bias
while reporting a number that sounds like diligence.

## E12 · Approaching an unfamiliar dataset

1. **Read the documentation first**, and write down what it does not say - licence, measurement
   method, collection dates.
2. **Establish what one row represents**, and check it arithmetically if you can.
3. **Shape and types**, so you know the size of the job.
4. **Missing, duplicated, impossible** - the cheap checks, which tell you about storage and nothing
   about meaning.
5. **One column at a time**: distributions, floors, ceilings, tails. Count rows at the maximum.
6. **Explain every extreme value before deciding what to do with it.**
7. **Then relationships**, at the right level of aggregation, and only then.
8. **Write the dictionary and the limitations**, including what you did not check.

**The step people skip is 1**, and the cost is that everything after it is uninterpretable. Without
knowing a row is a block group, the extreme averages look like errors, the target looks like a house
price, and the whole analysis answers a question nobody asked. Step 6 is second-most skipped and
costs you real signal - here, an entire column.

## E13 · Hospital ward-days

| This chapter's defect | The equivalent to look for |
|---|---|
| Row is a block group, not a house | A row is a **ward-day**, not a patient. Average length of stay is an average over patients discharged, and no patient-level conclusion follows |
| Censored target at $500,001 | Length of stay capped at a reporting limit ("30+ days"), or truncated by the extract's end date - patients still admitted have no length of stay yet |
| Ratio with a tiny denominator | Average length of stay on a day with two discharges; occupancy on a ward with four beds. Look for wards with very few patients |
| Three denominators in one row | Per ward-day (bed count), per patient (age, stay), per admission (a patient readmitted is two rows) |

**The arithmetic check for the unit of observation:** if the extract also carries a patient count and
a total bed-days figure, verify that `total bed-days / patients = average length of stay` exactly, as
`Population / AveOccup` recovered whole households here. If it comes out ragged, the denominators are
not what the column names claim, and finding that out on day one is worth more than any chart.

## E14 · For your manager

> "Nothing was missing, but four things were wrong. Each row is a whole neighbourhood rather than a
> house, so the file cannot price a house at all. Every value above five hundred thousand dollars was
> recorded as exactly five hundred thousand, and those are the most expensive neighbourhoods, so the
> model would be blind at the top of the market. Seventy-nine rows had averages taken over a handful
> of homes, and leaving them in made one useful column look worthless. And prices from 1990 describe
> a market that no longer exists. The day is what stopped us building something confident and wrong -
> it is written up so nobody has to spend it again."

118 words, no jargon, and the last sentence is the one that answers the actual question, which was
about the value of the time rather than about the data.

## E15 · The handover page

**California housing (1990 US census, via StatLib) - assessment for the pricing model**

*Four defects, with sizes.* (1) A row is a census block group, not a house; the target is already a
median over houses. (2) The target is censored at $500,001 - 965 rows, 4.68% - and the censored rows
are the richest (mean income 7.83 vs 3.68). (3) 79 rows carry averages over very few households,
which suppresses `AveOccup` (r -0.024 with them, -0.242 without) and halves `AveRooms` (0.152 vs
0.274). (4) `HouseAge` is censored at 52 (1,273 rows, 6.17%) and `MedInc` at 15.0001 (49 rows).

*Three columns not to use untreated.* `AveOccup` and `AveRooms` - set aside the 79 rows or model the
ratio explicitly. `HouseAge` - add a censoring indicator, and never use it without a location
variable, since it correlates +0.106 statewide but negatively inside six of eight regions.

*Recommendation.* **The dataset does not support the pricing model as asked.** The target is an area
median, not a house price, so no amount of accuracy makes it a valuer; and the data describes the
1990 market. It is suitable for one thing: learning how area-level features related to area-level
value in California in 1990. If the goal is pricing houses today, the correct next step is acquiring
current transaction-level data, and this file is worth a day of nobody's time beyond this note.

*The one-sentence version.* **"This data is about neighbourhoods in 1990, not houses today, so it
cannot price a house - but it did tell us that any model we build will need a location variable, or
it will conclude that older houses are worth more."**

## Where to go next

**Module 02 is finished.** The assessment in `assessments/` is next, and it is cumulative across all
eight chapters - work it without notes, then check.

After that, **module 03** starts the mathematics. It is the smallest amount that makes everything
afterwards readable, introduced where it is used rather than in advance, beginning with notation.